# Simulate Multilayer Dynamics

This notebook demonstrates how to simulate dynamical processes on multilayer networks using py3plex.

We'll cover:
* SIR epidemic models
* SIS dynamics
* Configuring parameters
* Analyzing results

## Installation

In [ ]:
!pip install py3plex -q

## Setup: Create a Multilayer Network

We'll create a two-layer network representing physical and digital contact layers.

In [ ]:
import numpy as np
from py3plex.core import multinet

# Create network
network = multinet.multi_layer_network(directed=False)

# Add nodes to both layers
nodes = []
for i in range(20):
    nodes.append({'source': i, 'type': 'physical'})
    nodes.append({'source': i, 'type': 'digital'})

network.add_nodes(nodes)

# Physical layer: ring structure
for i in range(20):
    network.add_edges([{
        'source': i,
        'target': (i + 1) % 20,
        'source_type': 'physical',
        'target_type': 'physical'
    }])

# Digital layer: random connections
rng = np.random.default_rng(42)
for i in range(30):
    source = rng.integers(0, 20)
    target = rng.integers(0, 20)
    if source != target:
        network.add_edges([{
            'source': source,
            'target': target,
            'source_type': 'digital',
            'target_type': 'digital'
        }])

print("Network created:")
network.basic_stats()

## SIR Epidemic Model

The SIR (Susceptible-Infected-Recovered) model simulates epidemic spread.

**Parameters:**
* `beta`: Infection probability per contact
* `gamma`: Recovery probability
* `initial_infected`: Fraction of initially infected nodes

In [ ]:
from py3plex.dynamics import SIRDynamics

# Create SIR dynamics
sir = SIRDynamics(
    network,
    beta=0.3,              # Infection rate
    gamma=0.1,             # Recovery rate
    initial_infected=0.05  # 5% initially infected
)

# Set seed for reproducibility
sir.set_seed(42)

# Run simulation
results = sir.run(steps=100)

print("SIR simulation completed")
print(f"Final state counts: {results.get_measure('state_counts')[-1]}")

## Visualize Epidemic Curve

Plot the time evolution of the epidemic.

In [ ]:
import matplotlib.pyplot as plt

# Get measures
prevalence = results.get_measure('prevalence')
state_counts = results.get_measure('state_counts')

# Extract S, I, R over time
susceptible = [counts.get('S', 0) for counts in state_counts]
infected = [counts.get('I', 0) for counts in state_counts]
recovered = [counts.get('R', 0) for counts in state_counts]

# Plot
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(prevalence, linewidth=2, color='red')
plt.xlabel('Time step')
plt.ylabel('Prevalence (fraction infected)')
plt.title('Epidemic Prevalence Over Time')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(susceptible, label='Susceptible', color='blue')
plt.plot(infected, label='Infected', color='red')
plt.plot(recovered, label='Recovered', color='green')
plt.xlabel('Time step')
plt.ylabel('Number of nodes')
plt.title('SIR Model State Evolution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Compare Different Parameters

Run simulations with different infection rates to see the impact.

In [ ]:
beta_values = [0.1, 0.2, 0.3, 0.4]
gamma = 0.1

plt.figure(figsize=(10, 6))

for beta in beta_values:
    sir = SIRDynamics(network, beta=beta, gamma=gamma, initial_infected=0.05)
    sir.set_seed(42)
    results = sir.run(steps=100)
    prevalence = results.get_measure('prevalence')
    
    plt.plot(prevalence, label=f'β={beta}', linewidth=2)

plt.xlabel('Time step')
plt.ylabel('Prevalence')
plt.title('Effect of Infection Rate (β) on Epidemic Spread')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Higher β (infection rate) leads to faster and larger epidemics")

## SIS Model (No Recovery)

The SIS (Susceptible-Infected-Susceptible) model allows reinfection.

In [ ]:
from py3plex.dynamics import SISDynamics

# Create SIS dynamics
sis = SISDynamics(
    network,
    beta=0.3,              # Infection rate
    gamma=0.1,             # Recovery rate
    initial_infected=0.05
)

sis.set_seed(42)
results = sis.run(steps=200)

# Plot
prevalence = results.get_measure('prevalence')

plt.figure(figsize=(10, 6))
plt.plot(prevalence, linewidth=2, color='purple')
plt.xlabel('Time step')
plt.ylabel('Prevalence')
plt.title('SIS Model: Endemic Equilibrium')
plt.grid(True, alpha=0.3)
plt.axhline(y=prevalence[-50:].mean(), color='red', linestyle='--', 
            label=f'Equilibrium ≈ {prevalence[-50:].mean():.3f}')
plt.legend()
plt.show()

print("SIS model reaches an endemic equilibrium where infection persists")

## Layer-Specific Analysis

Analyze epidemic spread within each layer separately.

In [ ]:
# Run simulation
sir = SIRDynamics(network, beta=0.3, gamma=0.1, initial_infected=0.05)
sir.set_seed(42)
results = sir.run(steps=100)

# Get final state
final_states = results.get_measure('node_states')[-1]

# Count infections by layer
physical_infected = sum(1 for (node, layer), state in final_states.items() 
                       if layer == 'physical' and state == 'I')
digital_infected = sum(1 for (node, layer), state in final_states.items() 
                      if layer == 'digital' and state == 'I')

print(f"Final infected nodes:")
print(f"  Physical layer: {physical_infected}")
print(f"  Digital layer: {digital_infected}")
print(f"\nThe digital layer's random connections facilitate faster spread")

## DynamicsBuilder API (Fluent Interface)

Use the fluent `D` builder for more expressive dynamics simulations.

In [ ]:
from py3plex.dynamics import D, SIR

# Use the fluent builder API
result = (
    D.simulate()
     .network(network)
     .process(SIR(beta=0.3, gamma=0.1))
     .initial_condition({node: 'S' for node in network.get_nodes()})
     .infect_random(n=2, state='I')  # Start with 2 random infected
     .steps(100)
     .seed(42)
     .run()
)

print("\nDynamicsBuilder API simulation completed!")
print(f"Final state counts: {result.final_state_counts()}")

# Plot the epidemic curve
import matplotlib.pyplot as plt
fig = result.plot_state_evolution()
plt.title("SIR Model - Built with Fluent API")
plt.show()

## Advanced SimulationResult Analysis

Extract detailed insights from simulation results.

In [ ]:
# Run a single simulation for detailed analysis
result = (
    D.simulate()
     .network(network)
     .process(SIR(beta=0.4, gamma=0.15))
     .initial_condition({node: 'S' for node in network.get_nodes()})
     .infect_random(n=2, state='I')
     .steps(80)
     .seed(42)
     .run()
)

# Analyze final states
print("\nFinal state distribution:")
final_counts = result.final_state_counts()
for state, count in sorted(final_counts.items()):
    print(f"  {state}: {count}")

# Get peak infection time and value
history = result.get_state_history()
infected_counts = [sum(1 for state in step.values() if state == 'I') for step in history]
peak_time = np.argmax(infected_counts)
peak_count = max(infected_counts)

print(f"\nPeak infection: {peak_count} nodes at time step {peak_time}")

# Compute attack rate (fraction ever infected)
total_nodes = len(network.get_nodes())
recovered_nodes = final_counts.get('R', 0)
attack_rate = recovered_nodes / total_nodes * 100
print(f"Attack rate: {attack_rate:.1f}% (nodes that were infected)")

## Uncertainty in Dynamics

Run multiple simulations to quantify uncertainty in outcomes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Run multiple simulations with different seeds
n_simulations = 20
results = []

for seed in range(n_simulations):
    result = (
        D.simulate()
         .network(network)
         .process(SIR(beta=0.3, gamma=0.1))
         .initial_condition({node: 'S' for node in network.get_nodes()})
         .infect_random(n=1, state='I')
         .steps(50)
         .seed(seed)
         .run()
    )
    results.append(result)

# Plot all trajectories with uncertainty bands
plt.figure(figsize=(10, 6))

# Extract state counts over time for all simulations
for result in results:
    history = result.get_state_history()
    infected_counts = [sum(1 for state in step.values() if state == 'I') for step in history]
    plt.plot(infected_counts, alpha=0.3, color='red')

# Compute and plot mean trajectory
all_infected = np.array([[sum(1 for state in step.values() if state == 'I') 
                          for step in result.get_state_history()] 
                         for result in results])
mean_infected = np.mean(all_infected, axis=0)
std_infected = np.std(all_infected, axis=0)

plt.plot(mean_infected, color='darkred', linewidth=2, label='Mean trajectory')
plt.fill_between(range(len(mean_infected)), 
                 mean_infected - std_infected, 
                 mean_infected + std_infected, 
                 alpha=0.3, color='red', label='±1 std dev')

plt.xlabel('Time Step')
plt.ylabel('Number of Infected')
plt.title(f'SIR Model Uncertainty ({n_simulations} simulations)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nPeak infection: {mean_infected.max():.1f} ± {std_infected[np.argmax(mean_infected)]:.1f}")

## Summary

In this tutorial, you learned:

* ✅ SIR epidemic model basics
* ✅ SIS model (no recovery)
* ✅ Parameter exploration and comparison
* ✅ Epidemic curve visualization
* ✅ DynamicsBuilder (D) fluent API
* ✅ SimulationResult detailed analysis
* ✅ Uncertainty quantification in dynamics
* ✅ Layer-specific dynamics analysis

## Next Steps

* Explore [RandomWalk dynamics](https://skblaz.github.io/py3plex/reference/dynamics.html)
* Try [custom dynamics models](https://skblaz.github.io/py3plex/guides/custom_dynamics.html)
* Learn about [community detection](community_detection.ipynb)
* Check out the [DSL tutorial](query_with_dsl.ipynb)